### 1) Configuration

In [0]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ==========================================================
# Enterprise Multi-Source Generator
# Configuration
# ==========================================================

from faker import Faker
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime, timedelta
import pandas as pd
import random
import uuid
import numpy as np

# ------------------------------------------------------------------
# Seeds (reproducible generation)
# ------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)

fake = Faker("pt_BR")

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

LANDING_ROOT = "/Volumes/landing/landing"
OLIST_ROOT = "/Volumes/landing/landing/olist_dev"

LOAD_DATE = "2026-07-21"

YEAR, MONTH, DAY = LOAD_DATE.split("-")
DATE_PATH = f"{YEAR}/{MONTH}/{DAY}"

# ------------------------------------------------------------------
# Source Systems Distribution
# ------------------------------------------------------------------

SYSTEM_DISTRIBUTION = {
    "crm": 1.00,          # 100%
    "erp": 0.90,          # 90%
    "ecommerce": 0.80,    # 80%
    "loyalty": 0.35       # 35%
}

# ------------------------------------------------------------------
# Output folders
# ------------------------------------------------------------------

OUTPUT_PATHS = {
    "crm_customers":
        f"{LANDING_ROOT}/crm/customers/{DATE_PATH}",

    "erp_customers":
        f"{LANDING_ROOT}/erp/customers/{DATE_PATH}",

    "erp_orders":
        f"{LANDING_ROOT}/erp/orders/{DATE_PATH}",

    "erp_payments":
        f"{LANDING_ROOT}/erp/payments/{DATE_PATH}",

    "ecommerce_customers":
        f"{LANDING_ROOT}/ecommerce/customers/{DATE_PATH}",

    "ecommerce_products":
        f"{LANDING_ROOT}/ecommerce/products/{DATE_PATH}",

    "ecommerce_sellers":
        f"{LANDING_ROOT}/ecommerce/sellers/{DATE_PATH}",

    "loyalty_customers":
        f"{LANDING_ROOT}/loyalty/customers/{DATE_PATH}"
}

# ------------------------------------------------------------------
# Data Quality Percentages
# ------------------------------------------------------------------

DQ = {

    "uppercase_name": 0.15,

    "abbreviated_name": 0.20,

    "missing_phone": 0.05,

    "missing_email": 0.03,

    "email_typo": 0.02,

    "missing_middle_name": 0.30

}

print("Configuration loaded.")

Configuration loaded.


### 2) Read Olist & Initial Profiling

In [0]:
# ==========================================================
# Read Olist Datasets
# ==========================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Reading Olist datasets...")
print("=" * 80)

customers = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/customers")
)

orders = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/orders")
)

order_items = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/order_items")
)

payments = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/order_payments")
)

reviews = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/order_reviews")
)

products = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/products")
)

sellers = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/sellers")
)

geolocation = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/geolocation")
)

categories = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/product_category_translation")
)

print("Datasets loaded successfully.")

Reading Olist datasets...
Datasets loaded successfully.


### Cell 3 — Dataset Profiling

In [0]:
# ==========================================================
# Dataset Profiling
# ==========================================================

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "categories": categories
}

summary = []

for name, df in datasets.items():

    summary.append(

        (
            name,
            df.count(),
            len(df.columns)
        )

    )

summary_df = spark.createDataFrame(
    summary,
    ["dataset", "rows", "columns"]
)

display(summary_df)

dataset,rows,columns
customers,99441,5
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,104162,7
products,32951,9
sellers,3095,4
geolocation,1000163,5
categories,71,2


### Cell 4 — Validate Business Keys

In [0]:
# ==========================================================
# Business Key Validation
# ==========================================================

print("=" * 80)
print("Customer statistics")
print("=" * 80)

display(

    customers.select(

        F.count("*").alias("rows"),

        F.countDistinct("customer_id").alias("customer_id"),

        F.countDistinct("customer_unique_id").alias("customer_unique_id")

    )

)

print("=" * 80)
print("Order statistics")
print("=" * 80)

display(

    orders.select(

        F.count("*").alias("rows"),

        F.countDistinct("order_id").alias("order_id"),

        F.countDistinct("customer_id").alias("customer_id")

    )

)

Customer statistics


rows,customer_id,customer_unique_id
99441,99441,96096


Order statistics


rows,order_id,customer_id
99441,99441,99441


In [0]:
# ==========================================================
# Build Master Customer Profile
# ==========================================================

print("=" * 80)
print("Creating Master Customer Profile...")
print("=" * 80)

# Only one record per real customer

master_seed = (
    customers
    .select(
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates(["customer_unique_id"])
    .orderBy("customer_unique_id")
)

print(f"Customers: {master_seed.count():,}")

master_pd = master_seed.toPandas()

print(f"Pandas rows: {len(master_pd):,}")

Creating Master Customer Profile...
Customers: 96,096
Pandas rows: 96,096


In [0]:
# ==========================================================
# Generate Canonical Customer Information
# ==========================================================

from faker import Faker
import random
import pandas as pd
import uuid

fake = Faker("pt_BR")

genders = ["Male", "Female"]

master_records = []

for idx, row in master_pd.iterrows():

    first = fake.first_name()

    middle = (
        fake.first_name()
        if random.random() < 0.40
        else None
    )

    last = fake.last_name()

    full = " ".join(
        [x for x in [first, middle, last] if x]
    )

    master_records.append({

        "global_customer_id":
            str(uuid.uuid4()),

        "customer_unique_id":
            row.customer_unique_id,

        "first_name":
            first,

        "middle_name":
            middle,

        "last_name":
            last,

        "full_name":
            full,

        "birth_date":
            fake.date_of_birth(
                minimum_age=18,
                maximum_age=80
            ),

        "gender":
            random.choice(genders),

        "cpf":
            fake.cpf(),

        "personal_email":
            fake.email(),

        "phone":
            fake.cellphone_number(),

        "street":
            fake.street_address(),

        "neighborhood":
            fake.bairro(),

        "city":
            row.customer_city,

        "state":
            row.customer_state

    })

master_profile_pd = pd.DataFrame(master_records)

print(master_profile_pd.head())

                     global_customer_id  ... state
0  1a17c964-5bf3-4f0f-8aa9-1c72703058c0  ...    SP
1  16873f4c-a05d-447a-8825-a61b3df17db6  ...    SP
2  8151aec0-1fb1-4d3e-875f-afc1239b67bd  ...    SC
3  678e389d-7ef1-4772-bfda-21da2949c18e  ...    PA
4  95359598-cf68-426f-9749-54882f0fdf9e  ...    SP

[5 rows x 15 columns]


In [0]:
# ==========================================================
# Convert Master Profile Back to Spark
# ==========================================================

master_profile = spark.createDataFrame(master_profile_pd)

display(master_profile.limit(10))

print("=" * 80)

print(f"Master Customers : {master_profile.count():,}")

print("=" * 80)

global_customer_id,customer_unique_id,first_name,middle_name,last_name,full_name,birth_date,gender,cpf,personal_email,phone,street,neighborhood,city,state
1a17c964-5bf3-4f0f-8aa9-1c72703058c0,0000366f3b9a7992bf8c76cfdf3221e2,Henry Gabriel,Benicio,Sousa,Henry Gabriel Benicio Sousa,1962-11-17,Male,321.564.087-26,alvesfrancisco@example.org,+55 92 98908-3863,"Ladeira de Abreu, 66",Itatiaia,cajamar,SP
16873f4c-a05d-447a-8825-a61b3df17db6,0000b849f77a49e4a4ce2b2a4ca5be3f,Leonardo,null,Cunha,Leonardo Cunha,2005-11-09,Female,518.302.467-71,siqueiramaria-clara@example.net,+55 53 96184-9593,"Área Ribeiro, 24",Barroca,osasco,SP
8151aec0-1fb1-4d3e-875f-afc1239b67bd,0000f46a3911fa3c0805444483337064,Ravi Lucca,Guilherme,Garcia,Ravi Lucca Guilherme Garcia,1963-01-24,Male,529.816.043-33,luanmartins@example.com,13 98350-3056,"Favela Nunes, 83",Morro Dos Macacos,sao jose,SC
678e389d-7ef1-4772-bfda-21da2949c18e,0000f6ccb0745a6a4b88665a16c9f078,Maria Cecília,null,Casa Grande,Maria Cecília Casa Grande,1962-03-29,Female,384.579.610-39,nicolemonteiro@example.org,+55 27 91226-9166,"Rua Theodoro Novaes, 2",Vila Jardim Alvorada,belem,PA
95359598-cf68-426f-9749-54882f0fdf9e,0004aac84e0df4da2b147fca70cf8255,Stella,Vitor,Vargas,Stella Vitor Vargas,1985-12-07,Female,146.930.258-60,piresana-cecilia@example.net,95 99325-2880,"Residencial Arthur Miguel Camargo, 44",Barreiro,sorocaba,SP
d5bc85e5-ccf9-4860-8857-4e296983e5f8,0004bd2a26a76fe21f786e4fbd80607f,Rodrigo,Emanuel,Barros,Rodrigo Emanuel Barros,1991-08-30,Female,182.953.647-82,valentinasa@example.net,+55 83 94657-8713,"Loteamento Almeida, 483",Ademar Maldonado,sao paulo,SP
ebd05aea-e16c-433c-bb5e-cbad8b459f39,00050ab1314c0e55a6ca13cf7181fecf,Diego,null,Santos,Diego Santos,1985-04-24,Male,310.627.895-12,valentinacardoso@example.net,86 93763-1165,"Quadra Cauê Rocha, 1",Nossa Senhora Da Conceição,campinas,SP
34f45ea5-7c8a-4e3e-901c-a4e153da3b5a,00053a61a98854899e70ed204dd4bafe,Camila,null,Caldeira,Camila Caldeira,1961-03-20,Male,387.196.240-69,macedoisis@example.org,+55 27 98013-2677,"Avenida Lucas Garcia, 7",Heliopolis,curitiba,PR
92836169-7ec9-464b-91eb-bdaee01e08dc,0005e1862207bf6ccc02e4228effd9a0,Maria,null,da Paz,Maria da Paz,1972-03-14,Female,872.196.053-40,danilo09@example.com,94 98208-1219,Feira de Guerra,Capitão Eduardo,teresopolis,RJ
b3d4677d-a41d-48e5-92d9-3bd1ec5ba7b4,0005ef4cd20d2893f0d9fbd94d3c0d97,Thales,Bruno,Pinto,Thales Bruno Pinto,1950-09-19,Female,985.216.734-00,ana-claramarques@example.com,+55 35 90799-1183,"Conjunto Cecília Azevedo, 55",Conjunto Capitão Eduardo,sao luis,MA


Master Customers : 96,096


In [0]:
# ==========================================================
# Generate System IDs
# ==========================================================

window = Window.orderBy("global_customer_id")

master_profile = (

    master_profile

    .withColumn(
        "seq",
        F.row_number().over(window)
    )

    .withColumn(
        "crm_customer_id",
        F.concat(
            F.lit("CRM"),
            F.lpad("seq",7,"0")
        )
    )

    .withColumn(
        "erp_customer_id",
        F.concat(
            F.lit("ERP"),
            F.lpad("seq",7,"0")
        )
    )

    .withColumn(
        "ecommerce_customer_id",
        F.concat(
            F.lit("WEB"),
            F.lpad("seq",7,"0")
        )
    )

    .withColumn(
        "loyalty_customer_id",
        F.concat(
            F.lit("LOY"),
            F.lpad("seq",7,"0")
        )
    )

    .drop("seq")

)

display(master_profile.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


global_customer_id,customer_unique_id,first_name,middle_name,last_name,full_name,birth_date,gender,cpf,personal_email,phone,street,neighborhood,city,state,crm_customer_id,erp_customer_id,ecommerce_customer_id,loyalty_customer_id
00003063-8505-4b03-9015-2ff96942751b,e19d02312392ed80fac4b7adeb01bdb1,Gabriela,João Felipe,Silveira,Gabriela João Felipe Silveira,1965-10-23,Female,154.629.703-07,gael09@example.net,+55 63 93960-5828,"Alameda Moraes, 12",Monte Azul,ibiuna,SP,CRM0000001,ERP0000001,WEB0000001,LOY0000001
0000aff8-3952-4067-8b7d-db0f73186e4d,651b9d14ef8acace2ba68cd88674ea73,Maria Liz,null,Cassiano,Maria Liz Cassiano,1960-09-26,Male,352.178.096-12,santosemilly@example.com,38 95730-7443,"Pátio de Fogaça, 59",Floramar,sorocaba,SP,CRM0000002,ERP0000002,WEB0000002,LOY0000002
0000e030-30dc-40c2-a87b-91921f9bf0dc,e8ff501d03c5f3ac7674c93dce33bb3e,Esther,null,Castro,Esther Castro,2004-02-07,Male,928.067.341-69,melomaria-julia@example.com,37 90015-6870,"Praça Alícia Ferreira, 98",Vila Boa Vista,joao pessoa,PB,CRM0000003,ERP0000003,WEB0000003,LOY0000003
00013fb9-a9b7-47f2-a213-59ca7f085fb2,4c0bac678bfb8ff0becd5b393b45a884,Nathan,null,Sales,Nathan Sales,1976-03-09,Female,903.872.564-74,araujozoe@example.com,95 95090-1920,Campo de Barros,Nazare,belo horizonte,MG,CRM0000004,ERP0000004,WEB0000004,LOY0000004
0002142a-72f9-4222-a8f7-c6625a2bf5a9,a0b7dc1fa38cb0c12f893dddc975cd2b,Marcelo,Daniel,Souza,Marcelo Daniel Souza,1959-07-14,Male,961.502.873-86,isabellypastor@example.net,+55 66 95439-6777,Ladeira Rhavi Mendonça,Indaiá,governador valadares,MG,CRM0000005,ERP0000005,WEB0000005,LOY0000005
0003b445-3d16-474f-978a-99680798012c,635aa3bbff322f693323857b56eff037,João Vitor,Raquel,da Mata,João Vitor Raquel da Mata,1986-03-24,Male,039.251.864-33,rporto@example.com,+55 77 94470-4937,"Setor de Abreu, 318",Vila Coqueiral,porto alegre,RS,CRM0000006,ERP0000006,WEB0000006,LOY0000006
0003cf04-36c1-4a94-bb63-ccb9116bea98,f24ee52da76941487bdac1de00300868,Gabriela,null,Santos,Gabriela Santos,1950-10-28,Male,593.462.178-55,castroravi@example.com,+55 43 90381-1003,"Conjunto de Andrade, 43",Mariquinhas,caxias do sul,RS,CRM0000007,ERP0000007,WEB0000007,LOY0000007
000409b4-663c-448b-a0eb-767b46dc7a86,8f1c56d4cc0b092ab6eef0328e0c7cd0,Maria Luiza,null,Mendonça,Maria Luiza Mendonça,1992-12-01,Female,379.105.426-07,kandrade@example.org,+55 86 98790-8033,"Aeroporto de Andrade, 7",Casa Branca,santo andre,SP,CRM0000008,ERP0000008,WEB0000008,LOY0000008
00040bd0-8859-49f5-a1b3-b01c488907b5,ff892cc741cb946aa3d909c57d905b88,Nicolas,null,Macedo,Nicolas Macedo,1990-04-22,Female,709.638.214-31,vitor-hugo59@example.com,34 94941-3246,"Jardim de Barbosa, 73",Pousada Santo Antonio,santa maria,RS,CRM0000009,ERP0000009,WEB0000009,LOY0000009
0005755b-c84f-4e65-9a1f-c29fff4e1946,a04d5a58a54312ea6d32516b88a2806f,Luiz Miguel,null,Gonçalves,Luiz Miguel Gonçalves,1970-07-25,Male,320.574.961-80,kevin38@example.net,75 90205-8701,"Favela de Novaes, 603",Glória,salto,SP,CRM0000010,ERP0000010,WEB0000010,LOY0000010


In [0]:
# ==========================================================
# Helper Functions
# Name Variations
# ==========================================================

import random
import string

def build_full_name(first, middle, last):

    parts = [first]

    if middle:
        parts.append(middle)

    parts.append(last)

    return " ".join(parts)


def random_name(first, middle, last):

    names = []

    names.append(build_full_name(first, middle, last))

    names.append(f"{first} {last}")

    if middle:
        names.append(f"{first} {middle[0]}. {last}")

    names.append(f"{first} {last[0]}.")

    names.append(f"{first[0]}. {last}")

    names.append(f"{first[0]} {last}")

    names.append(f"{last}, {first}")

    names.append(first)

    names.append(last)

    names.append(
        f"{first.lower()}{last.lower()}"
    )

    names.append(
        f"{first.lower()}_{last.lower()}"
    )

    names.append(
        f"{first.lower()}{random.randint(10,999)}"
    )

    return random.choice(names)

In [0]:
# ==========================================================
# Email Generator
# ==========================================================

FREE_EMAILS = [

    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com.br"

]


def random_email(first, last):

    domain = random.choice(FREE_EMAILS)

    options = [

        f"{first}.{last}@{domain}",

        f"{first}{last}@{domain}",

        f"{first[0]}{last}@{domain}",

        f"{first}{random.randint(10,999)}@{domain}",

        f"{first.lower()}_{last.lower()}@{domain}"

    ]

    return random.choice(options).lower()

In [0]:
# ==========================================================
# Phone Formats
# ==========================================================

def phone_variation(phone):

    phone = "".join(filter(str.isdigit, phone))

    if len(phone) < 11:
        return phone

    formats = [

        f"+55 ({phone[:2]}) {phone[2:7]}-{phone[7:]}",

        f"({phone[:2]}) {phone[2:7]}-{phone[7:]}",

        f"{phone}",

        f"{phone[:2]} {phone[2:]}"

    ]

    return random.choice(formats)

In [0]:
# ==========================================================
# Address Formatting
# ==========================================================

def address_variation(address):

    options = [

        address,

        address.replace("Rua","R."),

        address.replace("Avenida","Av."),

        address.replace("Travessa","Tv."),

        address.upper(),

        address.lower()

    ]

    return random.choice(options)

In [0]:
# ==========================================================
# Data Quality Simulation
# ==========================================================

def inject_name_issue(name):

    if random.random() < DQ["uppercase_name"]:

        return name.upper()

    if random.random() < DQ["abbreviated_name"]:

        pieces = name.split()

        if len(pieces) >= 2:

            return f"{pieces[0]} {pieces[-1][0]}."

    return name


def inject_email_issue(email):

    if random.random() > DQ["email_typo"]:

        return email

    return email.replace("@","@@",1)


def maybe_null(value, probability):

    if random.random() < probability:

        return None

    return value

In [0]:
# ==========================================================
# Customer Distribution
# ==========================================================

crm_seed = master_profile

erp_seed = master_profile.sample(
    False,
    SYSTEM_DISTRIBUTION["erp"],
    seed=SEED
)

ecommerce_seed = master_profile.sample(
    False,
    SYSTEM_DISTRIBUTION["ecommerce"],
    seed=SEED
)

loyalty_seed = master_profile.sample(
    False,
    SYSTEM_DISTRIBUTION["loyalty"],
    seed=SEED
)

print("CRM       :", crm_seed.count())
print("ERP       :", erp_seed.count())
print("WEB       :", ecommerce_seed.count())
print("LOYALTY   :", loyalty_seed.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CRM       : 96096
ERP       : 86532


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


WEB       : 77149
LOYALTY   : 33802


In [0]:
# ==========================================================
# Generate CRM Dataset
# ==========================================================

print("=" * 80)
print("Generating CRM...")
print("=" * 80)

crm_pd = crm_seed.toPandas()

crm_records = []

customer_status = [
    "ACTIVE",
    "INACTIVE",
    "PROSPECT"
]

marital_status = [
    "Single",
    "Married",
    "Divorced",
    "Widowed"
]

for _, c in crm_pd.iterrows():

    crm_records.append({

        "crm_customer_id": c.crm_customer_id,

        "customer_unique_id": c.customer_unique_id,

        "first_name": c.first_name,

        "middle_name": c.middle_name,

        "last_name": c.last_name,

        "full_name": build_full_name(
            c.first_name,
            c.middle_name,
            c.last_name
        ),

        "cpf": c.cpf,

        "birth_date": c.birth_date,

        "gender": c.gender,

        "marital_status": random.choice(marital_status),

        "profession": fake.job(),

        "annual_income": random.randint(
            25000,
            350000
        ),

        "email": random_email(
            c.first_name,
            c.last_name
        ),

        "phone": phone_variation(
            c.phone
        ),

        "street": address_variation(
            c.street
        ),

        "city": c.city,

        "state": c.state,

        "status": random.choice(customer_status),

        "risk_score": random.randint(0,100),

        "created_at": fake.date_between(
            start_date="-10y",
            end_date="-1y"
        ),

        "updated_at": fake.date_between(
            start_date="-365d",
            end_date="today"
        )

    })

crm_pd = pd.DataFrame(crm_records)

print(crm_pd.head())

Generating CRM...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  crm_customer_id                customer_unique_id  ...  created_at  updated_at
0      CRM0000001  e19d02312392ed80fac4b7adeb01bdb1  ...  2019-09-10  2025-10-23
1      CRM0000002  651b9d14ef8acace2ba68cd88674ea73  ...  2021-03-18  2025-08-22
2      CRM0000003  e8ff501d03c5f3ac7674c93dce33bb3e  ...  2019-08-13  2026-01-28
3      CRM0000004  4c0bac678bfb8ff0becd5b393b45a884  ...  2022-01-18  2025-11-01
4      CRM0000005  a0b7dc1fa38cb0c12f893dddc975cd2b  ...  2023-12-11  2025-08-15

[5 rows x 21 columns]


In [0]:
# ==========================================================
# CRM Data Quality
# ==========================================================

for i in crm_pd.index:

    crm_pd.at[i,"full_name"] = inject_name_issue(
        crm_pd.at[i,"full_name"]
    )

    crm_pd.at[i,"email"] = inject_email_issue(
        crm_pd.at[i,"email"]
    )

    crm_pd.at[i,"phone"] = maybe_null(
        crm_pd.at[i,"phone"],
        DQ["missing_phone"]
    )

    crm_pd.at[i,"middle_name"] = maybe_null(
        crm_pd.at[i,"middle_name"],
        DQ["missing_middle_name"]
    )

In [0]:
# ==========================================================
# CRM Spark DataFrame
# ==========================================================

crm = spark.createDataFrame(crm_pd)

print("=" * 80)

print("CRM Rows")

print(crm.count())

print("=" * 80)

display(crm.limit(10))

CRM Rows
96096


crm_customer_id,customer_unique_id,first_name,middle_name,last_name,full_name,cpf,birth_date,gender,marital_status,profession,annual_income,email,phone,street,city,state,status,risk_score,created_at,updated_at
CRM0000001,e19d02312392ed80fac4b7adeb01bdb1,Gabriela,null,Silveira,Gabriela João Felipe Silveira,154.629.703-07,1965-10-23,Female,Divorced,Ludomotricista,229613,gabrielasilveira@outlook.com,(55) 63939-605828,"Alameda Moraes, 12",ibiuna,SP,PROSPECT,58,2019-09-10,2025-10-23
CRM0000002,651b9d14ef8acace2ba68cd88674ea73,Maria Liz,null,Cassiano,Maria Liz Cassiano,352.178.096-12,1960-09-26,Male,Divorced,Manicure/pedicure,106427,maria liz.cassiano@outlook.com,38957307443,"Pátio de Fogaça, 59",sorocaba,SP,PROSPECT,31,2021-03-18,2025-08-22
CRM0000003,e8ff501d03c5f3ac7674c93dce33bb3e,Esther,null,Castro,Esther C.,928.067.341-69,2004-02-07,Male,Widowed,Professor de geofísica,274610,esthercastro@hotmail.com,+55 (37) 90015-6870,"Praça Alícia Ferreira, 98",joao pessoa,PB,ACTIVE,40,2019-08-13,2026-01-28
CRM0000004,4c0bac678bfb8ff0becd5b393b45a884,Nathan,null,Sales,Nathan Sales,903.872.564-74,1976-03-09,Female,Widowed,Pescador artesanal de peixes e camarões,40760,nsales@gmail.com,+55 (95) 95090-1920,Campo de Barros,belo horizonte,MG,ACTIVE,83,2022-01-18,2025-11-01
CRM0000005,a0b7dc1fa38cb0c12f893dddc975cd2b,Marcelo,null,Souza,Marcelo Daniel Souza,961.502.873-86,1959-07-14,Male,Single,Trabalhador na olericultura (legumes),282632,marcelo919@hotmail.com,55 66954396777,Ladeira Rhavi Mendonça,governador valadares,MG,PROSPECT,54,2023-12-11,2025-08-15
CRM0000006,635aa3bbff322f693323857b56eff037,João Vitor,Raquel,da Mata,João Vitor Raquel da Mata,039.251.864-33,1986-03-24,Male,Single,Maquinista de trem metropolitano,120548,joão vitor721@outlook.com,5577944704937,"Setor de Abreu, 318",porto alegre,RS,PROSPECT,17,2017-07-04,2025-10-04
CRM0000007,f24ee52da76941487bdac1de00300868,Gabriela,null,Santos,Gabriela Santos,593.462.178-55,1950-10-28,Male,Divorced,Operador de schutthecar,34931,gabrielasantos@outlook.com,+55 (55) 43903-811003,"CONJUNTO DE ANDRADE, 43",caxias do sul,RS,PROSPECT,70,2025-04-24,2026-06-11
CRM0000008,8f1c56d4cc0b092ab6eef0328e0c7cd0,Maria Luiza,null,Mendonça,Maria Luiza Mendonça,379.105.426-07,1992-12-01,Female,Single,Técnico de planejamento e programação da manutenção,48350,maria luiza_mendonça@yahoo.com.br,5586987908033,"Aeroporto de Andrade, 7",santo andre,SP,ACTIVE,44,2021-08-18,2025-12-23
CRM0000009,ff892cc741cb946aa3d909c57d905b88,Nicolas,null,Macedo,Nicolas M.,709.638.214-31,1990-04-22,Female,Widowed,Oficial do registro civil de pessoas jurídicas,52904,nicolasmacedo@outlook.com,34949413246,"Jardim de Barbosa, 73",santa maria,RS,PROSPECT,55,2022-08-09,2025-11-15
CRM0000010,a04d5a58a54312ea6d32516b88a2806f,Luiz Miguel,null,Gonçalves,Luiz Miguel Gonçalves,320.574.961-80,1970-07-25,Male,Widowed,Operador de área de corrida,196861,lgonçalves@hotmail.com,+55 (75) 90205-8701,"Favela de Novaes, 603",salto,SP,INACTIVE,22,2023-06-19,2025-08-05


In [0]:
# ==========================================================
# ERP Metrics
# ==========================================================

print("=" * 80)
print("Building ERP Metrics...")
print("=" * 80)

payments_value = (

    payments

    .groupBy("order_id")

    .agg(

        F.sum(
            F.col("payment_value").cast("double")
        ).alias("order_value")

    )

)

erp_metrics = (

    orders

    .join(
        payments_value,
        "order_id",
        "left"
    )

    .join(
        customers.select(
            "customer_id",
            "customer_unique_id"
        ),
        "customer_id"
    )

    .groupBy("customer_unique_id")

    .agg(

        F.countDistinct("order_id")
            .alias("total_orders"),

        F.min("order_purchase_timestamp")
            .alias("first_purchase"),

        F.max("order_purchase_timestamp")
            .alias("last_purchase"),

        F.sum("order_value")
            .alias("total_spent"),

        F.avg("order_value")
            .alias("average_ticket")

    )

)

display(erp_metrics.limit(10))

Building ERP Metrics...


customer_unique_id,total_orders,first_purchase,last_purchase,total_spent,average_ticket
c3d8a0b18ff1521752caf54292bdd5cf,1,2018-05-06 10:27:56,2018-05-06 10:27:56,339.09,339.09
4268ba27ab73c81466d3068e501223a3,1,2018-06-05 19:53:27,2018-06-05 19:53:27,27.29,27.29
ee3074a0ae884d6cfbe6a79a3f76f729,1,2018-05-23 12:40:38,2018-05-23 12:40:38,46.29,46.29
a7879b28606781a1488676b860e5b46a,1,2018-01-27 14:00:16,2018-01-27 14:00:16,198.2,198.2
4a17a49c025e9cfe067bca2ca537387c,1,2018-06-20 09:45:24,2018-06-20 09:45:24,213.0,213.0
641159b01ed966201b10aeb55df19208,1,2018-01-30 09:31:18,2018-01-30 09:31:18,87.64,87.64
0629b2e23c657cde1105b8282ba336ae,1,2018-02-20 10:27:28,2018-02-20 10:27:28,51.34,51.34
454c6584341c73204ed4f795e2946f56,1,2017-12-09 06:52:09,2017-12-09 06:52:09,182.69,182.69
6559799a5cf53662f7979eeb4a8b800f,1,2017-10-31 13:51:13,2017-10-31 13:51:13,156.73,156.73
d836cdcc6b2608a7f0ddf11e9b5bb2e5,1,2018-06-22 12:06:16,2018-06-22 12:06:16,368.99,368.99


In [0]:
# ==========================================================
# ERP Base
# ==========================================================

erp_base = (

    erp_seed

    .join(
        erp_metrics,
        "customer_unique_id",
        "left"
    )

)

display(erp_base.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_unique_id,global_customer_id,first_name,middle_name,last_name,full_name,birth_date,gender,cpf,personal_email,phone,street,neighborhood,city,state,crm_customer_id,erp_customer_id,ecommerce_customer_id,loyalty_customer_id,total_orders,first_purchase,last_purchase,total_spent,average_ticket
4c0bac678bfb8ff0becd5b393b45a884,00013fb9-a9b7-47f2-a213-59ca7f085fb2,Nathan,null,Sales,Nathan Sales,1976-03-09,Female,903.872.564-74,araujozoe@example.com,95 95090-1920,Campo de Barros,Nazare,belo horizonte,MG,CRM0000004,ERP0000004,WEB0000004,LOY0000004,2,2018-01-05 14:56:40,2018-01-05 14:56:40,866.2,433.1
ff892cc741cb946aa3d909c57d905b88,00040bd0-8859-49f5-a1b3-b01c488907b5,Nicolas,null,Macedo,Nicolas Macedo,1990-04-22,Female,709.638.214-31,vitor-hugo59@example.com,34 94941-3246,"Jardim de Barbosa, 73",Pousada Santo Antonio,santa maria,RS,CRM0000009,ERP0000009,WEB0000009,LOY0000009,1,2018-01-14 18:58:26,2018-01-14 18:58:26,70.03,70.03
a04d5a58a54312ea6d32516b88a2806f,0005755b-c84f-4e65-9a1f-c29fff4e1946,Luiz Miguel,null,Gonçalves,Luiz Miguel Gonçalves,1970-07-25,Male,320.574.961-80,kevin38@example.net,75 90205-8701,"Favela de Novaes, 603",Glória,salto,SP,CRM0000010,ERP0000010,WEB0000010,LOY0000010,1,2017-08-19 14:11:07,2017-08-19 14:11:07,151.18,151.18
635aa3bbff322f693323857b56eff037,0003b445-3d16-474f-978a-99680798012c,João Vitor,Raquel,da Mata,João Vitor Raquel da Mata,1986-03-24,Male,039.251.864-33,rporto@example.com,+55 77 94470-4937,"Setor de Abreu, 318",Vila Coqueiral,porto alegre,RS,CRM0000006,ERP0000006,WEB0000006,LOY0000006,1,2018-05-30 09:03:01,2018-05-30 09:03:01,123.84,123.84
f24ee52da76941487bdac1de00300868,0003cf04-36c1-4a94-bb63-ccb9116bea98,Gabriela,null,Santos,Gabriela Santos,1950-10-28,Male,593.462.178-55,castroravi@example.com,+55 43 90381-1003,"Conjunto de Andrade, 43",Mariquinhas,caxias do sul,RS,CRM0000007,ERP0000007,WEB0000007,LOY0000007,1,2017-03-27 16:14:25,2017-03-27 16:14:25,91.66,91.66
e19d02312392ed80fac4b7adeb01bdb1,00003063-8505-4b03-9015-2ff96942751b,Gabriela,João Felipe,Silveira,Gabriela João Felipe Silveira,1965-10-23,Female,154.629.703-07,gael09@example.net,+55 63 93960-5828,"Alameda Moraes, 12",Monte Azul,ibiuna,SP,CRM0000001,ERP0000001,WEB0000001,LOY0000001,1,2018-06-24 20:40:44,2018-06-24 20:40:44,107.72,107.72
651b9d14ef8acace2ba68cd88674ea73,0000aff8-3952-4067-8b7d-db0f73186e4d,Maria Liz,null,Cassiano,Maria Liz Cassiano,1960-09-26,Male,352.178.096-12,santosemilly@example.com,38 95730-7443,"Pátio de Fogaça, 59",Floramar,sorocaba,SP,CRM0000002,ERP0000002,WEB0000002,LOY0000002,1,2017-06-22 14:16:52,2017-06-22 14:16:52,102.03,102.03
8f1c56d4cc0b092ab6eef0328e0c7cd0,000409b4-663c-448b-a0eb-767b46dc7a86,Maria Luiza,null,Mendonça,Maria Luiza Mendonça,1992-12-01,Female,379.105.426-07,kandrade@example.org,+55 86 98790-8033,"Aeroporto de Andrade, 7",Casa Branca,santo andre,SP,CRM0000008,ERP0000008,WEB0000008,LOY0000008,1,2017-11-27 12:28:47,2017-11-27 12:28:47,123.89,123.89
e8ff501d03c5f3ac7674c93dce33bb3e,0000e030-30dc-40c2-a87b-91921f9bf0dc,Esther,null,Castro,Esther Castro,2004-02-07,Male,928.067.341-69,melomaria-julia@example.com,37 90015-6870,"Praça Alícia Ferreira, 98",Vila Boa Vista,joao pessoa,PB,CRM0000003,ERP0000003,WEB0000003,LOY0000003,1,2018-01-31 22:26:53,2018-01-31 22:26:53,63.05,63.05
a0b7dc1fa38cb0c12f893dddc975cd2b,0002142a-72f9-4222-a8f7-c6625a2bf5a9,Marcelo,Daniel,Souza,Marcelo Daniel Souza,1959-07-14,Male,961.502.873-86,isabellypastor@example.net,+55 66 95439-6777,Ladeira Rhavi Mendonça,Indaiá,governador valadares,MG,CRM0000005,ERP0000005,WEB0000005,LOY0000005,1,2018-02-07 14:27:34,2018-02-07 14:27:34,40.0,40.0


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ==========================================================
# ERP Dataset
# ==========================================================

erp_pd = erp_base.toPandas()

ACCOUNT_MANAGERS = [

    "Carlos Souza",
    "Ana Lima",
    "Juliana Costa",
    "Pedro Oliveira",
    "Marcos Santos",
    "Fernanda Rocha"

]

CUSTOMER_GROUP = [

    "Retail",

    "SMB",

    "Corporate",

    "Government"

]

PAYMENT_TERMS = [

    "NET15",

    "NET30",

    "NET45",

    "NET60"

]

TAX_CATEGORY = [

    "Normal",

    "Simples",

    "MEI"

]

erp_rows = []

for _, c in erp_pd.iterrows():

    erp_rows.append({

        "erp_customer_id":

            c.erp_customer_id,

        "customer_unique_id":

            c.customer_unique_id,

        "customer_name":

            random_name(

                c.first_name,

                c.middle_name,

                c.last_name

            ),

        "cpf":

            c.cpf,

        "tax_category":

            random.choice(TAX_CATEGORY),

        "customer_group":

            random.choice(CUSTOMER_GROUP),

        "payment_terms":

            random.choice(PAYMENT_TERMS),

        "credit_limit":

            random.randint(

                3000,

                250000

            ),

        "account_manager":

            random.choice(ACCOUNT_MANAGERS),

        "invoice_email":

            random_email(

                c.first_name,

                c.last_name

            ),

        "total_orders":

            c.total_orders,

        "first_purchase":

            c.first_purchase,

        "last_purchase":

            c.last_purchase,

        "total_spent":

            c.total_spent,

        "average_ticket":

            c.average_ticket

    })

erp_pd = pd.DataFrame(erp_rows)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ==========================================================
# ERP Dataset
# ==========================================================

erp_pd = erp_base.toPandas()

ACCOUNT_MANAGERS = [

    "Carlos Souza",
    "Ana Lima",
    "Juliana Costa",
    "Pedro Oliveira",
    "Marcos Santos",
    "Fernanda Rocha"

]

CUSTOMER_GROUP = [

    "Retail",

    "SMB",

    "Corporate",

    "Government"

]

PAYMENT_TERMS = [

    "NET15",

    "NET30",

    "NET45",

    "NET60"

]

TAX_CATEGORY = [

    "Normal",

    "Simples",

    "MEI"

]

erp_rows = []

for _, c in erp_pd.iterrows():

    erp_rows.append({

        "erp_customer_id":

            c.erp_customer_id,

        "customer_unique_id":

            c.customer_unique_id,

        "customer_name":

            random_name(

                c.first_name,

                c.middle_name,

                c.last_name

            ),

        "cpf":

            c.cpf,

        "tax_category":

            random.choice(TAX_CATEGORY),

        "customer_group":

            random.choice(CUSTOMER_GROUP),

        "payment_terms":

            random.choice(PAYMENT_TERMS),

        "credit_limit":

            random.randint(

                3000,

                250000

            ),

        "account_manager":

            random.choice(ACCOUNT_MANAGERS),

        "invoice_email":

            random_email(

                c.first_name,

                c.last_name

            ),

        "total_orders":

            c.total_orders,

        "first_purchase":

            c.first_purchase,

        "last_purchase":

            c.last_purchase,

        "total_spent":

            c.total_spent,

        "average_ticket":

            c.average_ticket

    })

erp_pd = pd.DataFrame(erp_rows)

In [0]:
# ==========================================================
# ERP Data Quality
# ==========================================================

for i in erp_pd.index:

    erp_pd.at[i,"customer_name"] = inject_name_issue(

        erp_pd.at[i,"customer_name"]

    )

    erp_pd.at[i,"invoice_email"] = maybe_null(

        erp_pd.at[i,"invoice_email"],

        0.10

    )

    erp_pd.at[i,"credit_limit"] = maybe_null(

        erp_pd.at[i,"credit_limit"],

        0.08

    )

In [0]:
# ==========================================================
# ERP Spark
# ==========================================================

erp = spark.createDataFrame(erp_pd)

print()

print("ERP Rows")

print(erp.count())

display(erp.limit(10))

In [0]:
# ==========================================================
# ERP Business Rules
# ==========================================================

def calculate_credit_limit(total_spent, total_orders):

    if total_spent is None:
        total_spent = 0

    if total_orders is None:
        total_orders = 0

    multiplier = random.uniform(2.0, 5.0)

    credit = max(
        3000,
        total_spent * multiplier
    )

    if total_orders > 20:
        credit *= 1.5

    return min(round(credit,2),250000)


def customer_group(total_spent):

    if total_spent is None:
        return "Prospect"

    if total_spent < 1000:
        return "Retail"

    if total_spent < 5000:
        return "Silver"

    if total_spent < 15000:
        return "Gold"

    return "Corporate"


def payment_terms(group):

    mapping = {

        "Prospect":"PREPAID",

        "Retail":"NET15",

        "Silver":"NET30",

        "Gold":"NET45",

        "Corporate":"NET60"

    }

    return mapping[group]


def choose_manager(group):

    managers = {

        "Prospect":[
            "Lucas Rocha",
            "Marina Costa"
        ],

        "Retail":[
            "Pedro Lima",
            "Juliana Alves"
        ],

        "Silver":[
            "Carlos Souza",
            "Fernanda Martins"
        ],

        "Gold":[
            "Ricardo Gomes",
            "Tatiane Melo"
        ],

        "Corporate":[
            "Eduardo Ribeiro",
            "Patricia Araujo"
        ]

    }

    return random.choice(managers[group])

In [0]:
# ==========================================================
# Ecommerce Reference Data
# ==========================================================

BROWSERS = [
    "Chrome",
    "Firefox",
    "Edge",
    "Safari",
    "Opera"
]

OPERATING_SYSTEMS = [
    "Windows",
    "Android",
    "iOS",
    "MacOS",
    "Linux"
]

DEVICES = [
    "Desktop",
    "Mobile",
    "Tablet"
]

MARKETING_SOURCE = [
    "Google",
    "Facebook",
    "Instagram",
    "Organic",
    "Referral",
    "LinkedIn",
    "TikTok"
]

LANGUAGES = [
    "pt-BR",
    "en-US",
    "es-ES"
]

In [0]:
# ==========================================================
# Favorite Product Category
# ==========================================================

favorite_category = (

    orders

    .join(order_items,"order_id")

    .join(products,"product_id")

    .join(
        customers.select(
            "customer_id",
            "customer_unique_id"
        ),
        "customer_id"
    )

    .groupBy(
        "customer_unique_id",
        "product_category_name"
    )

    .count()

)

window = Window.partitionBy(
    "customer_unique_id"
).orderBy(
    F.desc("count")
)

favorite_category = (

    favorite_category

    .withColumn(
        "rn",
        F.row_number().over(window)
    )

    .filter("rn=1")

    .drop("rn","count")

)

display(favorite_category.limit(10))

In [0]:
# ==========================================================
# Ecommerce Base
# ==========================================================

web_base = (

    ecommerce_seed

    .join(
        favorite_category,
        "customer_unique_id",
        "left"
    )

)

web_pd = web_base.toPandas()

In [0]:
# ==========================================================
# Ecommerce Dataset
# ==========================================================

print("=" * 80)
print("Generating Ecommerce...")
print("=" * 80)

web_rows = []

for _, c in web_pd.iterrows():

    display_name = random_name(
        c.first_name,
        c.middle_name,
        c.last_name
    )

    username = fake.user_name()

    browser = random.choice(BROWSERS)

    operating_system = random.choice(OPERATING_SYSTEMS)

    device = random.choice(DEVICES)

    marketing = random.choice(MARKETING_SOURCE)

    language = random.choice(LANGUAGES)

    newsletter = random.random() < 0.65

    account_created = fake.date_between(
        start_date="-8y",
        end_date="-30d"
    )

    last_login = fake.date_time_between(
        start_date="-30d",
        end_date="now"
    )

    wishlist = random.randint(0,40)

    carts_abandoned = random.randint(0,15)

    web_rows.append({

        "ecommerce_customer_id": c.ecommerce_customer_id,

        "customer_unique_id": c.customer_unique_id,

        "username": username,

        "display_name": display_name,

        "email_address": random_email(
            c.first_name,
            c.last_name
        ),

        "phone_number": phone_variation(c.phone),

        "browser": browser,

        "operating_system": operating_system,

        "device": device,

        "marketing_source": marketing,

        "preferred_language": language,

        "newsletter_optin": newsletter,

        "preferred_category": c.product_category_name,

        "wishlist_items": wishlist,

        "abandoned_carts": carts_abandoned,

        "account_created": account_created,

        "last_login": last_login

    })

web_pd = pd.DataFrame(web_rows)

print(web_pd.head())

In [0]:
# ==========================================================
# Ecommerce Data Quality
# ==========================================================

for i in web_pd.index:

    web_pd.at[i,"display_name"] = inject_name_issue(
        web_pd.at[i,"display_name"]
    )

    web_pd.at[i,"email_address"] = inject_email_issue(
        web_pd.at[i,"email_address"]
    )

    web_pd.at[i,"phone_number"] = maybe_null(
        web_pd.at[i,"phone_number"],
        0.08
    )

    if random.random() < 0.05:

        web_pd.at[i,"preferred_category"] = None

In [0]:
# ==========================================================
# Ecommerce Spark
# ==========================================================

web = spark.createDataFrame(web_pd)

print()

print("Ecommerce Customers")

print(web.count())

display(web.limit(10))

In [0]:
# ==========================================================
# Ecommerce Products
# ==========================================================

products_pd = products.toPandas()

BRANDS = [

    "Samsung",
    "LG",
    "Apple",
    "Philips",
    "Bosch",
    "Electrolux",
    "Dell",
    "Lenovo",
    "Nike",
    "Adidas",
    "Tramontina"

]

for i in products_pd.index:

    products_pd.at[i,"brand"] = random.choice(BRANDS)

    products_pd.at[i,"supplier_code"] = f"SUP-{random.randint(1000,9999)}"

    products_pd.at[i,"is_active"] = random.random() > 0.03

    products_pd.at[i,"launch_year"] = random.randint(2018,2024)

products_web = spark.createDataFrame(products_pd)

In [0]:
# ==========================================================
# Ecommerce Sellers
# ==========================================================

sellers_pd = sellers.toPandas()

for i in sellers_pd.index:

    sellers_pd.at[i,"seller_rating"] = round(
        random.uniform(3.5,5.0),
        2
    )

    sellers_pd.at[i,"premium_partner"] = (
        random.random() < 0.25
    )

    sellers_pd.at[i,"account_manager"] = random.choice([

        "Carlos Souza",

        "Ana Lima",

        "Fernanda Costa",

        "Pedro Santos"

    ])

products_web.printSchema()

sellers_web = spark.createDataFrame(sellers_pd)

In [0]:
# ==========================================================
# Loyalty Business Rules
# ==========================================================

def calculate_points(total_spent):

    if total_spent is None:
        return 0

    return int(total_spent * 10)


def redeemed_points(points):

    if points == 0:
        return 0

    percentage = random.uniform(0.10, 0.70)

    return int(points * percentage)


def loyalty_tier(points):

    if points < 1000:
        return "Bronze"

    if points < 5000:
        return "Silver"

    if points < 15000:
        return "Gold"

    return "Platinum"


def wallet_balance(points):

    return round(points * 0.01,2)

In [0]:
# ==========================================================
# Loyalty Base
# ==========================================================

loyalty_base = (

    loyalty_seed

    .join(
        erp_metrics,
        "customer_unique_id",
        "left"
    )

)

display(loyalty_base.limit(10))

loyalty_pd = loyalty_base.toPandas()

In [0]:
# ==========================================================
# Loyalty Dataset
# ==========================================================

PREFERRED_STORE = [

    "São Paulo",

    "Rio de Janeiro",

    "Curitiba",

    "Belo Horizonte",

    "Brasília"

]

CAMPAIGNS = [

    "Black Friday",

    "Summer Sale",

    "Birthday",

    "Cashback",

    "Christmas"

]

loyalty_rows = []

for _, c in loyalty_pd.iterrows():

    points = calculate_points(
        c.total_spent
    )

    redeemed = redeemed_points(
        points
    )

    loyalty_rows.append({

        "loyalty_customer_id":

            c.loyalty_customer_id,

        "customer_unique_id":

            c.customer_unique_id,

        "member_name":

            random_name(

                c.first_name,

                c.middle_name,

                c.last_name

            ),

        "join_date":

            fake.date_between(

                start_date="-8y",

                end_date="-30d"

            ),

        "points":

            points,

        "redeemed_points":

            redeemed,

        "available_points":

            points - redeemed,

        "tier":

            loyalty_tier(points),

        "wallet_balance":

            wallet_balance(points),

        "preferred_store":

            random.choice(

                PREFERRED_STORE

            ),

        "last_campaign":

            random.choice(

                CAMPAIGNS

            ),

        "active_member":

            random.random() < 0.95

    })

loyalty_pd = pd.DataFrame(loyalty_rows)

In [0]:
# ==========================================================
# Loyalty Data Quality
# ==========================================================

for i in loyalty_pd.index:

    loyalty_pd.at[i,"member_name"] = inject_name_issue(

        loyalty_pd.at[i,"member_name"]

    )

    if random.random() < 0.03:

        loyalty_pd.at[i,"preferred_store"] = None

    if random.random() < 0.02:

        loyalty_pd.at[i,"wallet_balance"] = None

In [0]:
# ==========================================================
# Loyalty Spark
# ==========================================================

loyalty = spark.createDataFrame(loyalty_pd)

print()

print("Loyalty Members")

print(loyalty.count())

display(loyalty.limit(10))

In [0]:
# ==========================================================
# Validation
# ==========================================================

datasets = {

    "CRM": crm,

    "ERP": erp,

    "WEB": web,

    "LOYALTY": loyalty

}

summary = []

for name, df in datasets.items():

    summary.append(

        (

            name,

            df.count(),

            len(df.columns)

        )

    )

display(

    spark.createDataFrame(

        summary,

        [

            "system",

            "rows",

            "columns"

        ]

    )

)

In [0]:
# ==========================================================
# Write Landing
# ==========================================================

datasets = {

    OUTPUT_PATHS["crm_customers"]: crm,

    OUTPUT_PATHS["erp_customers"]: erp,

    OUTPUT_PATHS["erp_orders"]: orders,

    OUTPUT_PATHS["erp_payments"]: payments,

    OUTPUT_PATHS["ecommerce_customers"]: web,

    OUTPUT_PATHS["ecommerce_products"]: products_web,

    OUTPUT_PATHS["ecommerce_sellers"]: sellers_web,

    OUTPUT_PATHS["loyalty_customers"]: loyalty

}

for path, df in datasets.items():

    print(f"Writing {path}")

    (

        df.write

        .mode("overwrite")

        .parquet(path)

    )

print()

print("Landing successfully generated.")